# 02 - PHA, FMEA and HAZOP: structured hazard identification

**Week 2; suggested duration: two teaching periods**

## Learning objectives

- Create a preliminary hazard list
- Develop the FMEA failure-mode/effect/cause chain
- Calculate RPN and recognize its limitations
- Develop HAZOP deviations using guidewords


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)


## 1. Three methods, three perspectives

- **PHA:** what hazards exist at a high level?
- **FMEA:** how can a component or function fail, and what is the effect?
- **HAZOP:** what deviations from design intent can occur, what causes them, and what follows?

The objective is traceability between the methods, not three unrelated lists.


## 2. PHA: the initial TK-101 hazard list

PHA is an early, high-level analysis. Examples include overflow, ignition/fire and loss of containment. Priority categories are organization-specific; the labels below are for teaching only.


In [ ]:
pha=pd.DataFrame([
 {"hazard":"Overflow","cause":"Level protection ineffective","consequence":"Release","safeguard":"LSHH+XV+operator","priority":"high"},
 {"hazard":"Fire","cause":"Release + ignition","consequence":"Fire escalation","safeguard":"containment+fire protection","priority":"high"},
 {"hazard":"Loss of containment","cause":"Tank/connection integrity loss","consequence":"Release","safeguard":"inspection+containment","priority":"medium"}])
pha


## 3. FMEA and RPN

A typical FMEA chain is **function -> failure mode -> effect -> cause -> control -> rating -> action**. The classic teaching RPN is:

$$RPN=S\times O\times D.$$

Define the rating scales before use. **Fixed RPN thresholds are not universal acceptance limits.** Equal RPN values can hide very different severity profiles.


In [ ]:
def rpn(S,O,D):
    if not all(1<=int(x)<=10 for x in [S,O,D]): raise ValueError("S/O/D: 1..10")
    return int(S)*int(O)*int(D)

examples=pd.DataFrame([
 {"mode":"A","S":10,"O":2,"D":4},
 {"mode":"B","S":5,"O":4,"D":4},
 {"mode":"C","S":8,"O":5,"D":2}])
examples["RPN"]=examples.apply(lambda r:rpn(r.S,r.O,r.D),axis=1)
examples


### Equal RPN does not mean equal risk profile

`10 x 2 x 4` and `5 x 4 x 4` both equal 80. In a safety context, severity can warrant independent priority. Compare a severity-first ranking.


In [ ]:
examples.sort_values(["S","O","D","RPN"],ascending=False)


## 4. TK-101 FMEA

The S/O/D ratings are **synthetic teaching values**, not operational evidence. This is a compact worksheet: a full engineering study would also record causes, controls, actions, ownership and review status.


In [ ]:
fmea=pd.DataFrame([
 {"component":"LT-101","failure_mode":"failed low / bias","effect":"high level detected late","S":9,"O":4,"D":6},
 {"component":"LSHH-101","failure_mode":"fail on demand","effect":"trip not initiated","S":10,"O":2,"D":7},
 {"component":"XV-101","failure_mode":"fail on demand","effect":"inflow continues","S":10,"O":2,"D":6},
 {"component":"HMI/operator","failure_mode":"alarm missed","effect":"no timely manual recovery","S":9,"O":3,"D":5}])
fmea["RPN"]=fmea.apply(lambda r:rpn(r.S,r.O,r.D),axis=1)
fmea.sort_values("RPN",ascending=False)


## 5. HAZOP: guideword + parameter -> deviation

Study `LEVEL` and `FLOW` at the `TK-101 filling` node. Example guidewords are NO, MORE, LESS, REVERSE and OTHER THAN. Not every combination is meaningful; the method supports structured engineering reasoning.


In [ ]:
hazop=pd.DataFrame([
 {"node":"TK-101 filling","parameter":"LEVEL","guideword":"MORE","deviation":"High level","causes":"excess inflow; LT failed low; shutdown failure","consequences":"overflow/release","safeguards":"LAH; LSHH; XV; operator"},
 {"node":"TK-101 filling","parameter":"FLOW","guideword":"MORE","deviation":"Excess feed flow","causes":"pump/control fault","consequences":"level rises faster","safeguards":"flow/level monitoring; trip"},
 {"node":"TK-101 shutdown","parameter":"SHUTDOWN","guideword":"NO","deviation":"No effective shutdown","causes":"LSHH/XV fail on demand","consequences":"inflow continues at high level","safeguards":"operator intervention"}])
hazop


## 6. Traceability between the three methods

The following chain provides candidate inputs to fault-tree analysis:

**PHA: Overflow -> FMEA: LT failed low / LSHH FOD / XV FOD -> HAZOP: MORE LEVEL / NO SHUTDOWN -> FTA basic events.**


In [ ]:
trace=pd.DataFrame([
 {"PHA hazard":"Overflow","FMEA mode":"LT-101 failed low","HAZOP deviation":"MORE LEVEL","next":"FTA basic event"},
 {"PHA hazard":"Overflow","FMEA mode":"LSHH fail on demand","HAZOP deviation":"NO SHUTDOWN","next":"FTA basic event"},
 {"PHA hazard":"Overflow","FMEA mode":"XV fail on demand","HAZOP deviation":"NO SHUTDOWN","next":"FTA basic event"}])
trace


## Try it!

1. Add an FMEA row for maintenance overdue. Is it a component failure or an organizational/process failure mode?
2. Find two equal-RPN cases with different severities.
3. Add a LESS FLOW or REVERSE FLOW HAZOP row where physically meaningful.
4. Identify which safeguards should appear in the next fault tree.


## Summary

PHA structures hazards, FMEA structures failure modes and HAZOP structures deviations. Next, develop the combinational logic: **which event combinations are sufficient for the overflow top event?**
